# All D11 Calculation Pathways

Evaluate all 32 slab-level `D11` pathways on ECTP slabs, export the paper-ready distribution
figure, and assemble the representative-pathways table used in the manuscript.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from notebook_utils import create_ectp_slabs, hess_rcparams, load_pits
from paper_figure_utils import (
    build_d11_distribution_figure,
    notebook_latex,
    prepare_d11_top_pathways_table,
    save_paper_figure,
)
from snowpyt_mechparams.pathway import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from snowpyt_mechparams.graph import default_graph

hess_rcparams()


ModuleNotFoundError: No module named 'notebook_utils'

## Load Data and Execute All D11 Paths


In [ ]:
pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)
pathways = find_parameterizations(default_graph, default_graph.get_node('D11'))

engine = ExecutionEngine()
config = ExecutionConfig(include_method_uncertainty=False)

pathway_slab_success = {}
pathway_rel_unc = {}
pathway_nominal = {}

for slab in ectp_slabs:
    results = engine.execute_all(slab, 'D11', config=config)
    for pathway_result in results.pathways.values():
        density_method = pathway_result.methods_used.get('density', 'unknown')
        emod_method = pathway_result.methods_used.get('elastic_modulus', 'unknown')
        nu_method = pathway_result.methods_used.get('poissons_ratio', 'unknown')
        pathway_key = f'{density_method} -> {emod_method} -> {nu_method}'

        pathway_slab_success.setdefault(pathway_key, 0)
        pathway_rel_unc.setdefault(pathway_key, [])
        pathway_nominal.setdefault(pathway_key, [])

        d11_nominal = None
        d11_std = None
        for trace in pathway_result.computation_trace:
            if trace.parameter == 'D11' and trace.success and trace.output is not None:
                output = trace.output
                if hasattr(output, 'nominal_value'):
                    d11_nominal = float(output.nominal_value)
                    d11_std = float(output.std_dev)
                else:
                    d11_nominal = float(output)
                    d11_std = 0.0
                break

        if d11_nominal is not None:
            pathway_slab_success[pathway_key] += 1
            pathway_nominal[pathway_key].append(d11_nominal)
            if d11_nominal != 0 and d11_std is not None:
                pathway_rel_unc[pathway_key].append(d11_std / abs(d11_nominal))

ordered_paths = sorted(pathway_slab_success.items(), key=lambda item: item[1], reverse=True)

summary_rows = []
for pathway_key, n_success in ordered_paths:
    nominal_values = np.asarray(pathway_nominal[pathway_key], dtype=float)
    rel_unc = np.asarray(pathway_rel_unc[pathway_key], dtype=float)
    summary_rows.append(
        {
            'Pathway': pathway_key,
            'Slabs': int(n_success),
            'Coverage (%)': f'{100.0 * n_success / total_slabs:.1f}',
            'Mean D11 (N m)': f'{np.mean(nominal_values):.3e}',
            'Mean relative uncertainty (%)': f'{100.0 * np.mean(rel_unc):.1f}',
        }
    )

d11_summary = pd.DataFrame(summary_rows)
print(f'Loaded {len(pits):,} pits, {total_slabs:,} ECTP slabs, and {len(pathways)} D11 pathways')
d11_summary.head(12)


## Export Figure and Representative Table


In [ ]:
d11_paths = save_paper_figure(
    build_d11_distribution_figure(ordered_paths, pathway_nominal, total_slabs, top_n=12),
    'd11_top_pathways',
    close=True,
)
top_pathways_table = prepare_d11_top_pathways_table(
    d11_summary,
    ordered_paths,
    pathway_nominal,
    top_n=12,
)
print('D11 figure exports:', d11_paths)
top_pathways_table


## Copy-Ready LaTeX Table


In [ ]:
print(notebook_latex(top_pathways_table))
